# Neurotransmitter Probability Variance across Drosophila Neuropils

This projects aims to answer - how do neurotransmitter probability distributions vary across neuropils in Drosophila?
The datasets should be downloaded into the data directory following the instructions on GitHub. 

## Stage 0: Set up environment

In [5]:
%load_ext autoreload 
%autoreload 2

# Import external libraries
import dask
from IPython.core.display import HTML
import pyvista as pv
from dask.distributed import Client

# Import core python libraries
import os

# Import local scripts (brainz.py, pipeline.py, plotting.py, 
# preprocess.py, util.py)
from scripts import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Set up in-line 3D plot rendering
pv.set_jupyter_backend("client")
HTML("""
<style>
.output_svg {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")

In [6]:
# Set globals
OUTDIR = os.path.join(os.path.dirname(__name__), "results", "notebook")
if not os.path.exists(OUTDIR): os.mkdir(OUTDIR)
MINSIZE = 30 # Minimum number of nodes in a cluster/community

In [7]:
allow_bytes = pipeline.get_ram_allowance()
PARTITION_SIZE = pipeline.get_partition_size(num_threads = 4, allow_bytes = allow_bytes)
client = pipeline.start_dask(num_threads = 4, allow_bytes = allow_bytes)
dask.config.set({"dataframe.shuffle.method": "tasks"})
preprocess.run() # ~10 mins first run

15.8 GB available on machine; allowing 7.9 GB

Dask Dashboard at http://192.168.0.133:55604/status

Notice: this may take some time if this is the first time running preprocess.py.


10:08:03 Preprocessing complete!


C:\Users\cielb\anaconda3\envs\nta1\Lib\site-packages\distributed\node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 55604 instead
  warnings.warn(
2026-05-30 10:09:25,358 - distributed.worker.memory - WARNING - Worker is at 84% memory usage. Pausing worker.  Process memory: 6.68 GiB -- Worker memory limit: 7.90 GiB
2026-05-30 10:09:25,679 - distributed.worker.memory - WARNING - Worker is at 77% memory usage. Resuming worker. Process memory: 6.14 GiB -- Worker memory limit: 7.90 GiB
2026-05-30 10:09:53,145 - distributed.worker.memory - WARNING - Worker is at 81% memory usage. Pausing worker.  Process memory: 6.45 GiB -- Worker memory limit: 7.90 GiB
2026-05-30 10:09:53,385 - distributed.worker.memory - WARNING - Worker is at 71% memory usage. Resuming worker. Process memory: 5.67 GiB -- Worker memory limit: 7.90 GiB
2026-05-30 10:10:11,847 - distributed.worker.memory - WARNING - Worker is at 84% memory usage

## Stage 1: Load Data

This stage of the pipeline loads and merges data from file, creating a single cohesive dataframe containing metadata for every synapse belonging to a neuron-neuron connection in proofread_connections_783.parquet. 

Neurotransmitter probability normalisation is required because the probabilities of some points do not sum to exactly 1 across all 6 neurotransmitters, but Dirichlet regression and composition analysis requires these probabilities to sum to exactly 1. Additionally, neurotransmitter normalisation reduces the data from 6 compositional variables to 3 compositional variables: GABA, an inhibitory neurotransmitter; Acetylcholine ('ach'), an excitatory neurotransmitter, and 'other', which encompasses the remaining neurotransmitters which cannot be reliably classified as solely inhibitory or excitatory (rather, they are usually neuromodulators).

Although the proofread connections file contains a compact representation of synapse data already (synapse count, and the same neurotransmitter probabilities allocated to each synapse), this stage of the pipeline expands the compact representation back to representing one synapse per row. This is because compositional analysis of neurotransmitter probabilities across synapses (e.g., in Dirichlet regression and in plotting ternary plots) requires this expanded format. 

In [8]:
# Choose dataset to use. Note the decisions made in this notebook are based
# on the full connectome dataset (proofread_connections_783.parquet). The
# other datasets are selections of different neuropils, and are provided for
# debugging/testing purposes.

DATASET = "data/proofread_connections_783.parquet" # ~ 1.05 GB

#DATASET = "data/large.parquet" # ~ 583 MB
#DATASET = "data/medium.parquet" # ~ 272 MB
#DATASET = "data/small.parquet" # ~ 56 MB
#DATASET = "data/tiny.parquet" # ~ 215 KB

In [9]:
connectome = pipeline.load_connectome(DATASET, PARTITION_SIZE)
connectome.head(3).style.hide(axis="index")

10:08:11 Loading connectome ...
10:08:16 Connectome loaded


pre,post,neuropil,gaba,ach,glut,oct,ser,da
720575940629970489,720575940631267655,AVLP_R,0.654330,0.023704,0.272418,0.048125,0.000472,0.000951
720575940623828999,720575940612348950,SLP_R,0.386517,0.024020,0.580512,0.000817,0.000857,0.007278
720575940624078484,720575940616950161,SMP_R,0.001719,0.979256,0.001811,0.000016,0.005870,0.011328


In [10]:
connectome = pipeline.normalise_nt_probs(connectome)
connectome.head(3).style.hide(axis="index")

10:08:19 Normalising neurotransmitter probabilities ...
10:08:19 Neurotransmitter probabilities normalised


pre,post,neuropil,gaba,ach,other
720575940629970489,720575940631267655,AVLP_R,0.654330,0.023704,0.321967
720575940623828999,720575940612348950,SLP_R,0.386517,0.024020,0.589463
720575940624078484,720575940616950161,SMP_R,0.001719,0.979256,0.019025


In [11]:
# This step takes a little longer because it merges the 1 GB 
# proofread_synapses_783.parquet file with the ~13 GB 
# flywire_synapses_783.parquet file. 
# Took ~20 min with ~8 GB RAM allowance.
connectome = pipeline.attach_synapse_coords(connectome, PARTITION_SIZE)
connectome.head(3).style.hide(axis="index")

10:08:26 Attaching coordinates ...


C:\Users\cielb\anaconda3\envs\nta1\Lib\site-packages\dask\dataframe\dask_expr\_collection.py:318: UserWarning: Dask annotations {'resources': {'shuffle': 1}} detected. Annotations will be ignored when using query-planning.
  warnings.warn(


10:09:04 Coordinates attached


index,pre,post,neuropil,gaba,ach,other,x,y,z
0,720575940625002936,720575940632028216,GNG,0.646454,0.073814,0.279732,881336,509456,111960
1,720575940625002936,720575940632028216,GNG,0.646454,0.073814,0.279732,852464,515064,84360
2,720575940625002936,720575940632028216,GNG,0.646454,0.073814,0.279732,872544,515156,108480


In [12]:
connectome = pipeline.attach_neuropil_metadata(connectome)
connectome.head(3).style.hide(axis="index")

10:12:06 Attaching neuropil metadata ...
10:12:06 Neuropil metadata attached


index,pre,post,neuropil,gaba,ach,other,x,y,z,region,neuropil_desc
0,720575940625002936,720575940632028216,GNG,0.646454,0.073814,0.279732,881336,509456,111960,Gnathal Ganglia,gnathal ganglia
1,720575940625002936,720575940632028216,GNG,0.646454,0.073814,0.279732,852464,515064,84360,Gnathal Ganglia,gnathal ganglia
2,720575940625002936,720575940632028216,GNG,0.646454,0.073814,0.279732,872544,515156,108480,Gnathal Ganglia,gnathal ganglia


## Stage 2: Identify & Visualise Neurotransmitter Clusters

This step identifies frequent cluster compositions and allocates each point to either a frequent cluster or labels it as 'noise'. A brain map is then plotted, with each point's colour corresponding to its allocated neurotransmitter cluster. Although this approach requires extra effort and will be harder to automate in the future, it is superior to the naive approach of labelling each point on a brain map by the neurotransmitter with the highest probability for that edge. This is because the latter approach ignores important compositional information and can lump two very different neurotransmitter ratios into the same group (e.g., a point with 100% acetylcholine probability would be labelled in the same manner as a point with 34% acetylcholine, 33% gaba, and 33% other probabilities).

### Visualise Initial Neurotransmitter Probability Distribution

In [ ]:
# Sample ~1 million points to speed up render (using matplotlib)
sample = pipeline.downsample(connectome, 1_000_000, allow_bytes)
filename = os.path.join(OUTDIR, "init_overall_distribution.svg")
plotting.plot_overall_distribution(sample, "Neurotransmitter Probability Frequencies Across Drosophila Connectome", filename)

### Cluster Synapses By Neurotransmitter Probabilities

In the above plot, there are k clusters.

In [ ]:
connectome = cluster_nt_probs
connectome.head(3).style.hide(axis="index")

### Visualise Neurotransmitter Cluster Distributions Across The Drosophila Brain

In [ ]:
# Plot brain map with 500,000 points
plotter = brainz.get_plotter(clustered, "hdbscan_id")
brainz.save(plotter, OUTDIR, _id="clusterd_brain_map")
print("Saved brain map!")
plotter.show()

In [ ]:
plotter.close() # Free GPU memory

## Preview - Can Nodule Neurons be Isolated from Linker Neurons?

Here I visualise the initial output generated from HDBSCAN clustering on xyz coordinates on a small sample of the drosophila connectome. The aim is to ensure the clustering parameters assign cluster IDs to groups in a way that approximates real nodules. As the 3D plot shows, the Drosophila connectome cannot be segregated into 'nodules' using this approach. Indeed, most synapses are equidistant from each other in 3D space. To isolate 'nodule'-like structures and generate visualisations such as those seen in the literature, filtering by cell type is required, however, cell type annotations are not present in this dataset. Interestingly, there do appear to be some patches of yellow, blue, and pink, indicating there may still be some differentiation in neurotransmitter probabilities across neuropils. The remainder of this analysis will focus on comparing neurotransmitter probabilities across neuropils using the unclustered dataframe.

In [ ]:
connectome = pipeline.load_connectome("data/tiny.parquet")
connectome = pipeline.normalise_nt_probs(connectome)
connectome = pipeline.attach_synapse_coords(connectome)
condensed = pipeline.condense(connectome)
condensed = util.do_hdbscan(condensed, MINSIZE)
clustered = pipeline.extend(condensed, connectome)

In [ ]:
# Plot 500,000 points
plotter = brainz.get_plotter(clustered, "hdbscan_id")
brainz.save(plotter, OUTDIR, _id="clusterd_brain_map")
print("Saved brain map!")
plotter.show()

In [ ]:
plotter.close() # Free memory

## Visualise Neurotransmitter Probability Distributions

### Overall Neurotransmitter Probability Distributions

In [ ]:
# Sample ~1 million points to speed up render (using matplotlib)
#sample = util.downsample(connectome, 1_000_000)
sample = connectome
filename = os.path.join(OUTDIR, "overall_distribution.svg")
plotting.plot_overall_distribution(sample, "Full Dataset", filename)

### Get Neuropil Summary Statistics

In [ ]:
neuropils = pipeline.get_neuropil_summary_stats(connectome)
neuropils.head(10)

### Neuropil Probability Distributions with respect to Size (Number of Synapses)

In [ ]:
filename = os.path.join(OUTDIR, "mean_neuropil_probs.svg")
plotting.plot_mean_nt_probs_by_neuropil_size(neuropils, filename)

### Neurotransmitter Probability Variance by Neuropil Size

In [ ]:
filename = os.path.join(OUTDIR, "neuropil_nt_prob_variance_by_size.svg")
plotting.plot_variance_by_neuropil_size(neuropils, filename)

### Neuropil Probability Distributions

In [ ]:
filename = os.path.join(OUTDIR, "neuropil_probability_distributions.svg")
plotting.plot_hex_per_neuropil(connectome, filename)

## Statistical Analysis

I want to know whether neurotransmitter probabilities are different between neuropils. Because probabilities are a form of compositional data (the sum of the variables is 1, and each variable is bounded between 0 and 1), the Dirichlet distribution is suitable. This statistical analysis is a simple one based on average synapse probabilities within neuropils. A more statistically robust method would involve using the 'other' column to calculate the ranges of excitatory and inhibitory probabilities for each synaptic connection, then running a statistical analysis that can handle comparing range values within a bounded interval, e.g., Bayesian Dirichlet regression with interval priors. The Dirichlet function I used can only operate on points, not ranges. There does not appear to be an out-of-box Dirichlet regression function in any Python libraries, so I will interface with the R DirichletReg package via rpy2.

In [ ]:
fitted, model_summary, null_summary, anova_result = do_stats.run(connectome)